# Relationships in SLayer

TL;DR: Use dotted relationship-path syntax everywhere — to reference metrics/dimensions from joined datasets inside queries, AND inside the `sql` of dimension, metric, and dataset-filter definitions. The `__` spelling is only an emitted-SQL alias detail, never an input form; the legacy `__` split-alias input form is a hard error. 

Relationships connect datasets so that dimensions and metrics from one dataset are accessible when querying another. SLayer only supports **LEFT JOINs** — the kind most commonly used for data enrichment.

If you think of left joins as directed edges of a graph whose vertices are datasets, then the dimensions, metrics, and filters in a dataset have access to fields from any dataset reachable in the relationship graph.

This notebook illustrates every aspect of relationships with working code.

See also: [Datasets — Relationships](../../concepts/datasets.md#relationships) | [Queries — Cross-Dataset Metrics](../../concepts/queries.md#cross-dataset-metrics) | [Ingestion — Diamond Relationships](../../concepts/ingestion.md#diamond-relationships)

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, datasets = ensure_jaffle_shop()

## Basic Relationships

A relationship is defined by a **target dataset** and a list of **key pairs** — column pairs to join on. The `orders` dataset has two relationships, auto-generated from its foreign keys to `customers` and `stores`:

In [2]:
orders_dataset = next(d for d in datasets if d.name == "orders")

print("orders dataset relationships:")
for r in orders_dataset.relationships:
    print(f"  -> {r.to}  ON {r.column_pairs}")

# Query using a joined dimension
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(orders.*)", "sum(orders.order_total)"],
        "dimensions": ["customers.name"],
        "order": [{"field": "orders.order_total_sum", "direction": "desc"}],
        "limit": 5,
    }
)

print("\nTop 5 customers by revenue:")
for row in result.data:
    print(f"  {row['customers.name']}: {row['orders._count']} orders, ${row['orders.order_total_sum']:,.2f}")

orders dataset relationships:
  -> customers  ON [('customer_id', 'id')]
  -> stores  ON [('store_id', 'id')]

Top 5 customers by revenue:
  Theodore Anderson: 155 orders, $14,552.41
  Rhonda Gomez: 152 orders, $14,212.12
  Joshua Wells: 146 orders, $13,596.26
  Shane Yates: 144 orders, $13,460.59
  Jennifer Garcia: 140 orders, $13,036.55


## Referencing Joined Datasets in Queries (Dot Syntax)

SLayer uses **dot syntax** to reference dimensions and metrics from joined datasets:

- 1-hop: `customers.name` (orders -> customers)
- Multi-hop: `orders.customers.name` (items -> orders -> customers)

Every reference starts at a dataset, reached from the population by its only to-one path; spelling the route from there (`orders.customers.name`, or `items.orders.customers.name` from the population) avoids ambiguity when there are multiple ways to reach a dataset.

In [3]:
# 1-hop: orders -> stores
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(orders.*)"],
        "dimensions": ["stores.name"],
    }
)

print("1-hop (orders -> stores):")
for row in result.data:
    print(f"  {row['stores.name']}: {row['orders._count']} orders")

1-hop (orders -> stores):
  Brooklyn: 252474 orders
  Chicago: 104104 orders
  New Orleans: 16134 orders
  Philadelphia: 192098 orders
  San Francisco: 92264 orders


In [4]:
# Multi-hop: items -> orders -> customers
result = engine.execute_sync(
    query={
        "population": "items",
        "metrics": ["count(items.*)"],
        "dimensions": ["orders.customers.name"],
        "order": [{"field": "items._count", "direction": "desc"}],
        "limit": 5,
    }
)

print("Multi-hop (items -> orders -> customers):")
for row in result.data:
    print(
        f"  {row['orders.customers.name']}: {row['items._count']} line items"
    )

Multi-hop (items -> orders -> customers):
  Theodore Anderson: 1550 line items
  Rhonda Gomez: 1520 line items
  Lori Butler: 1467 line items
  Joshua Wells: 1460 line items
  Shane Yates: 1440 line items


## Referencing Joined Datasets in SQL Snippets

When defining dimensions, metrics, and filters at the dataset level (in YAML or Python), their `sql` fields use the **same dotted relationship-path syntax** as queries. SLayer resolves the path through the relationship graph and emits the correct table aliases in the generated SQL.

| Context | Syntax | Example |
|---------|--------|---------|
| Query dimensions/metrics | dots | `orders.customers.name` |
| Dataset SQL expressions | dots | `orders.customers.name` |

The generated SQL uses a `__`-delimited internal alias for each joined table (`LEFT JOIN customers AS orders__customers`), but that spelling is an emitted-SQL detail — you always write dotted paths. The legacy `__` split-alias input form (`orders__customers.name` meant as a relationship walk) is no longer accepted; it is a hard error.

In [5]:
# Dataset-level SQL uses the same dotted relationship-path syntax as queries.
# Auto-ingestion doesn't generate these — joined fields are resolved at query time.
#
# Here's a manual example:

from slayer.core.models import DatasetField

manual_field = DatasetField(
    name="customer_name",
    expression="orders.customers.name",  # a path through a relationship
    datatype="String",
)
print(f"Dataset-level SQL (dotted path): sql={manual_field.sql!r}")
print("Query-level (same dotted path):   orders.customers.name")

# SLayer resolves the dotted path to the correct __ internal alias in generated SQL:
result = engine.execute_sync(
    query={
        "population": "items",
        "metrics": ["count(items.*)"],
        "dimensions": ["orders.customers.name"],
        "limit": 3,
    }
)

print(f"\nResult keys: {list(result.data[0].keys())}")
for row in result.data:
    print(f"  {row['orders.customers.name']}: {row['items._count']} items")


Dataset-level SQL (dotted path): sql=None
Query-level (same dotted path):   orders.customers.name

Result keys: ['orders.customers.name', 'items._count']
  Aaron Daniels: 79 items
  Aaron Diaz: 195 items
  Aaron Ellis: 301 items


## Auto-Ingesting Schemas

When auto-ingesting a database, SLayer introspects FK constraints and creates a `DatasetRelationship` for each direct FK on the source table. Multi-hop reachability (e.g. `items → orders → customers`) is resolved at query time by walking each intermediate dataset's own relationships — it is not baked into the source dataset's relationship list.

In [6]:
import sqlalchemy as sa

from setup_jaffle import DB_PATH
from slayer.core.models import DatasourceConfig

ds = DatasourceConfig(name="jaffle_shop", type="duckdb", database=DB_PATH)
sa_engine = sa.create_engine(ds.resolve_env_vars().get_connection_string())
inspector = sa.inspect(sa_engine)
print("Foreign keys:")
for table in sorted(inspector.get_table_names()):
    refs = sorted({fk["referred_table"] for fk in inspector.get_foreign_keys(table)})
    if refs:
        print(f"  {table} -> {refs}")
sa_engine.dispose()

items_dataset = next(d for d in datasets if d.name == "items")
orders_dataset = next(d for d in datasets if d.name == "orders")

print("\nitems relationships (direct FKs only):")
for r in items_dataset.relationships:
    src, tgt = r.column_pairs[0]
    print(f"  -> {r.to:<12} ON {src:<20} = {tgt}")

print("\norders relationships (direct FKs only):")
for r in orders_dataset.relationships:
    src, tgt = r.column_pairs[0]
    print(f"  -> {r.to:<12} ON {src:<20} = {tgt}")

print("\nMulti-hop: items reaches customers via orders.relationships")

Foreign keys:
  items -> ['orders', 'products']
  orders -> ['customers', 'stores']


  stores -> ['regions']
  supplies -> ['products']
  tweets -> ['customers']

items relationships (direct FKs only):
  -> orders       ON order_id             = id
  -> products     ON sku                  = sku

orders relationships (direct FKs only):
  -> customers    ON customer_id          = id
  -> stores       ON store_id             = id

Multi-hop: items reaches customers via orders.relationships


## Diamond Relationships

A **diamond relationship** occurs when the same table is reachable via multiple FK paths. For example:

```
orders -> customers -> regions
orders -> warehouses -> regions
```

SLayer treats each path as a separate copy of the target table, using **path-based aliases** to disambiguate:
- `customers.regions.name` -> table alias `customers__regions`  
- `warehouses.regions.name` -> table alias `warehouses__regions`

The Jaffle Shop schema doesn't have a natural diamond, so let's construct another schema for illustration:

In [7]:
from relationships_utils import setup_diamond_example

diamond_engine, diamond_storage, diamond_datasets, diamond_db_path, diamond_work_dir = setup_diamond_example()

diamond_orders = next(d for d in diamond_datasets if d.name == "orders")
diamond_customers = next(d for d in diamond_datasets if d.name == "customers")
diamond_warehouses = next(d for d in diamond_datasets if d.name == "warehouses")

print("orders relationships (direct FKs only):")
for r in diamond_orders.relationships:
    src, tgt = r.column_pairs[0]
    print(f"  -> {r.to:<12} ON {src:<25} = {tgt}")

print("\ncustomers relationships:")
for r in diamond_customers.relationships:
    src, tgt = r.column_pairs[0]
    print(f"  -> {r.to:<12} ON {src:<25} = {tgt}")

print("\nwarehouses relationships:")
for r in diamond_warehouses.relationships:
    src, tgt = r.column_pairs[0]
    print(f"  -> {r.to:<12} ON {src:<25} = {tgt}")

print("\nDiamond: orders reaches regions via customers.relationships AND warehouses.relationships")
print("Each path gets a unique alias: customers__regions vs warehouses__regions")

orders relationships (direct FKs only):
  -> customers    ON customer_id               = id
  -> warehouses   ON warehouse_id              = id

customers relationships:
  -> regions      ON region_id                 = id

warehouses relationships:
  -> regions      ON region_id                 = id

Diamond: orders reaches regions via customers.relationships AND warehouses.relationships
Each path gets a unique alias: customers__regions vs warehouses__regions


In [8]:
# Query both paths simultaneously
result = diamond_engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(orders.*)", "sum(orders.amount)"],
        "dimensions": [
            "customers.regions.name",
            "orders.warehouses.regions.name",
        ],
    }
)

print(f"{'Customer Region':<18} {'Warehouse Region':<18} {'Orders':>7} {'Amount':>10}")
print("-" * 56)
for row in result.data:
    cr = row["customers.regions.name"]
    wr = row["orders.warehouses.regions.name"]
    print(f"{cr:<18} {wr:<18} {row['orders._count']:>7} ${row['orders.amount_sum']:>9,.2f}")

Customer Region    Warehouse Region    Orders     Amount
--------------------------------------------------------
Central            Central                  1 $   150.00
Central            West                     1 $   225.00
East               Central                  2 $   375.00
East               West                     2 $   600.00
West               East                     3 $   750.00
West               West                     1 $   175.00


### Recombining Diamond Relationships with Filters

By default, each path to the same table produces independent copies. If you want to enforce that the two paths resolve to the same row (re-creating a true diamond), add a query filter equating the two paths — filters take the same dotted relationship-path syntax as dimensions:

If you want to make that constraint permanent, add the same expression to the `filters` of the dataset in question (`orders` in the example below) instead of to each query.

In [9]:
# Only keep orders where customer and warehouse are in the same region.
result = diamond_engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(orders.*)", "sum(orders.amount)"],
        "dimensions": ["customers.regions.name"],
        "filter": ["customers.regions.name = orders.warehouses.regions.name"],
    }
)

print("Orders where customer and warehouse are in the same region:")
for row in result.data:
    print(f"  {row['customers.regions.name']}: {row['orders._count']} orders, ${row['orders.amount_sum']:,.2f}")

Orders where customer and warehouse are in the same region:
  Central: 1 orders, $150.00
  West: 1 orders, $175.00


## Dynamic Relationships (query extensions)

Relationships can be added at query time via a query's `extensions`, without modifying the stored dataset. This is useful for:
- Ad-hoc relationships to lookup tables
- Joining to datasets created dynamically from queries
- Adding context-specific enrichment

More on this in the post (and companion notebook) on [multistage queries](../06_multistage_queries/multistage_queries.md).

## Summary

SLayer's relationship system provides:

| Feature | Description |
|---------|-------------|
| **LEFT JOINs** | The only join type, used for data enrichment |
| **Dot syntax** | `customers.name`, `orders.customers.regions.name` in queries |
| **Generated `__` aliases** | Internal table aliases in emitted SQL (`... AS orders__customers`); you always write dotted paths |
| **Auto-ingestion** | FK constraints become `DatasetRelationship` objects automatically |
| **Diamond relationships** | Same table via multiple paths gets separate path-based aliases |
| **Dynamic relationships** | A query's `extensions` add relationships at query time without modifying datasets |

See the [Datasets docs](../../concepts/datasets.md#relationships) and [Ingestion docs](../../concepts/ingestion.md) for the full reference.